# Confirm relative quality on test, and probe federation scale

Selection happened on the validation split; this evaluates the chosen
variant once on held-out test. It also extends the scaling sweep to
K = 20 and K = 32, where the paper currently stops at 12.


In [ ]:
import glob, os, shutil, sys
REPO='/kaggle/working/repo'
hits=glob.glob('/kaggle/input/**/paper/build_paper.py', recursive=True)
assert hits, 'attach the aha-fed-code dataset'
src=os.path.dirname(os.path.dirname(hits[0]))
if not os.path.isdir(os.path.join(REPO,'paper')):
    shutil.copytree(src, REPO, dirs_exist_ok=True)
os.chdir(REPO); sys.path.insert(0, REPO)
print('staged:', os.getcwd())


In [ ]:
!pip install -q medmnist opacus


In [ ]:
import subprocess, sys
PROBE=('import torch;'
       'a=torch.randn(4,3,8,8,device="cuda");'
       'w=torch.randn(2,3,3,3,device="cuda");'
       'torch.nn.functional.conv2d(a,w).sum().item();'
       'print(torch.cuda.get_device_name(0), torch.__version__)')
r=subprocess.run([sys.executable,'-c',PROBE],capture_output=True,text=True)
print(r.stdout.strip() or r.stderr.strip()[-400:])
if r.returncode!=0:
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    'torch==2.4.1','torchvision==0.19.1',
                    '--index-url','https://download.pytorch.org/whl/cu118'],check=True)
    r=subprocess.run([sys.executable,'-c',PROBE],capture_output=True,text=True)
    print(r.stdout.strip() or r.stderr.strip()[-400:])
    if r.returncode!=0: raise SystemExit('no working torch/GPU combination')


## Relative quality on the held-out test split


In [ ]:
from paper.run_real_experiments import run_one, OUT_DIR

SEEDS=[42,123,456,7,13,21,99,2024,31337,555]
PNEU={'dataset':'pneumoniamnist','num_classes':2,'samples_per_client':300}

for s in SEEDS:
    run_one(OUT_DIR/f'pneu_relq_s{s}.json', aggregator='adaptive',
            alpha=1.0, beta=4.0, gamma=4.0, relative_quality=True,
            seed=s, num_rounds=20, **PNEU)
print('test-split confirmation done')


## Federation scale beyond K = 12


In [ ]:
for nc in [20, 32]:
    spc = max(200, 4000 // nc)
    common = {'dataset':'pathmnist','num_classes':9,'num_clients':nc,
              'samples_per_client':spc,'noisy_client':nc-1}
    for s in SEEDS:
        suffix='' if s==42 else f'_s{s}'
        run_one(OUT_DIR/f'real_scale_fedavg_k{nc}{suffix}.json',
                aggregator='fedavg', seed=s, **common)
        run_one(OUT_DIR/f'real_scale_aha_k{nc}{suffix}.json',
                aggregator='adaptive', alpha=1.0, beta=4.0, gamma=4.0,
                seed=s, **common)
    print(f'K={nc} done')


In [ ]:
import glob, json, statistics as st
from pathlib import Path
from scipy.stats import wilcoxon

def fin(f):
    d=json.loads(Path(f).read_text()); return d.get('final') or {}
def by_seed(pat):
    import re
    out={}
    for f in sorted(glob.glob(pat)):
        m=re.search(r'_s(\d+)$', Path(f).stem)
        out[m.group(1) if m else '42']=fin(f)
    return out

print('=== PneumoniaMNIST, held-out test ===')
r=by_seed('runs/paper/real/pneu_relq_s*.json')
a=by_seed('runs/paper/real/pneu_aha_fed_s*.json')
c=sorted(set(r)&set(a))
if c:
    for m in ['ece','accuracy','auroc']:
        x=[r[k][m] for k in c]; y=[a[k][m] for k in c]
        print(f'  {m:9s} relative {st.mean(x):.4f}  absolute {st.mean(y):.4f}  '
              f'p={wilcoxon(x,y).pvalue:.4f}')

print()
print('=== scaling ===')
for k in ['4','8','12','20','32']:
    ah=by_seed(f'runs/paper/real/real_scale_aha_k{k}_s*.json')
    ah.update(by_seed(f'runs/paper/real/real_scale_aha_k{k}.json'))
    fa=by_seed(f'runs/paper/real/real_scale_fedavg_k{k}_s*.json')
    fa.update(by_seed(f'runs/paper/real/real_scale_fedavg_k{k}.json'))
    cc=sorted(set(ah)&set(fa))
    if not cc: continue
    x=[ah[i]['ece'] for i in cc]; y=[fa[i]['ece'] for i in cc]
    p=wilcoxon(x,y).pvalue if len(cc)>=6 else float('nan')
    print(f'  K={k:<3} AHA {st.mean(x):.4f}  FedAvg {st.mean(y):.4f}  '
          f'{st.mean(y)/st.mean(x):.2f}x  p={p:.4f}  (n={len(cc)})')


In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/relq_scale_results.zip runs/paper/real
!ls -lh /kaggle/working/relq_scale_results.zip
